# Retry: exact cert timeouts

Seven graphs in `exact_certs.csv` have `lcr_cert` or `ucr_cert` equal to `timed_out`. They fall into two categories:

- **Large automorphism group** (n=12–120): `_certify_exact` hangs computing `character_table()` for groups of order up to 83M. Fix: find a small subgroup Γ ≤ Aut whose restriction to the eigenspace still has multiplicity-1 irreps, then certify with Γ. Since Γ ≤ Aut, the Γ-orbit constraints refine the Aut-orbit constraints, so the certificate verifies against full Aut-orbits too.
- **Large n, pexpect overflow** (n=220, 234): iterating conjugacy classes through Sage's pexpect GAP interface overflows for large permutation degrees. Fix: use libgap directly. HoG 56133 additionally has very large eigenspaces (dim 110 and 88) but rational eigenvalues, so we compute the eigenspace over ℚ for speed.

In [1]:
import csv, json, os, sys, time
import random as pyrandom
from sage.libs.gap.libgap import libgap
from sage.arith.misc import prime_divisors

sys.path.insert(0, os.path.abspath('../../cr'))
sys.setrecursionlimit(50000)  # needed for deep algebraic arithmetic in large eigenspaces (e.g. HoG 56133 dim=88)
from orbits import get_edge_orbits
load('../../cr/exact_cert.sage')

pari.allocatemem(1 << 32)  # 4 GB — needed for HoG 42921 (PARI algebraic arithmetic)

DATA_PATH  = os.path.abspath('../raw_hog_data.jsonl')
OUTPUT_CSV = 'exact_certs.csv'
PHI_JSONL  = 'exact_certs.jsonl'

# Load g6 strings
g6_map = {}
with open(DATA_PATH) as f:
    for line in f:
        d = json.loads(line)
        g6_map[d['id']] = d['g6']

# Load CSV; identify rows where a direction timed out
with open(OUTPUT_CSV) as f:
    reader = csv.DictReader(f)
    fieldnames = reader.fieldnames
    all_rows = {int(r['id']): r for r in reader}

timeout_rows = [
    r for r in all_rows.values()
    if r['lcr_cert'] == 'timed_out' or r['ucr_cert'] == 'timed_out'
]
print(f"{len(timeout_rows)} graphs to retry:")
for r in timeout_rows:
    print(f"  HoG {r['id']}  n={r['order']}  lcr_cert={r['lcr_cert']}  ucr_cert={r['ucr_cert']}")

PARI stack size set to 4294967296 bytes, maximum size set to 4294967296
7 graphs to retry:
  HoG 32782  n=12  lcr_cert=  ucr_cert=timed_out
  HoG 32866  n=12  lcr_cert=  ucr_cert=timed_out
  HoG 2693  n=28  lcr_cert=timed_out  ucr_cert=
  HoG 50838  n=40  lcr_cert=  ucr_cert=timed_out
  HoG 51415  n=120  lcr_cert=timed_out  ucr_cert=reg_bipartite
  HoG 56133  n=220  lcr_cert=timed_out  ucr_cert=timed_out
  HoG 42921  n=234  lcr_cert=timed_out  ucr_cert=reg_bipartite


## Subgroup search helpers

We search for the smallest Γ ≤ Aut with multiplicity-1 on the target eigenspace using three strategies: random k-generator subgroups, the derived series of Aut, and Sylow p-subgroups.

In [2]:
def _gap_perm_to_sage(g_gap, aut_group):
    domain = sorted(aut_group.domain())
    zero_based = (domain[0] == 0)
    sp = g_gap.sage()
    cycles = sp.to_cycles(singletons=False)
    if zero_based:
        cycles = [tuple(x - 1 for x in cyc) for cyc in cycles]
    return aut_group(cycles)


def find_subgroup_candidates(G, aut_full, n_random=100, max_k=4, seed_base=0):
    """Return (aut_orbits, candidates) sorted by subgroup order."""
    aut_orbits = get_edge_orbits(G, list(aut_full.gens()))
    candidates = []

    def _record(name, gens):
        gens = [g for g in gens if not g.is_one()]
        if not gens:
            return
        try:
            Gamma = aut_full.subgroup(gens)
            order = int(Gamma.order())
        except Exception:
            return
        orbs = get_edge_orbits(G, gens)
        candidates.append({'name': name, 'group': Gamma, 'gens': gens,
                            'orbits': orbs, 'order': order})

    for k in range(2, max_k + 1):
        for seed in range(n_random):
            pyrandom.seed(int(seed_base * 10000 + seed * 100 + k))
            gens_r, tries = [], 0
            while len(gens_r) < k and tries < 50:
                g = aut_full.random_element()
                if not g.is_one():
                    gens_r.append(g)
                tries += 1
            if len(gens_r) == k:
                _record(f'rand/k={k},s={seed}', gens_r)

    try:
        cur = libgap(aut_full)
        for depth in range(1, 8):
            cur = cur.DerivedSubgroup()
            if int(cur.Order()) <= 1:
                break
            gens_d = [_gap_perm_to_sage(g, aut_full) for g in cur.GeneratorsOfGroup()]
            _record(f'derived[{depth}]', gens_d)
    except Exception as e:
        print(f"  derived series error: {e}")

    try:
        gap_aut = libgap(aut_full)
        for p in prime_divisors(int(aut_full.order())):
            try:
                syl = gap_aut.SylowSubgroup(int(p))
                gens_s = [_gap_perm_to_sage(g, aut_full) for g in syl.GeneratorsOfGroup()]
                _record(f'sylow/p={p}', gens_s)
            except Exception:
                pass
    except Exception as e:
        print(f"  Sylow error: {e}")

    seen, unique = set(), []
    for c in sorted(candidates, key=lambda c: c['order']):
        if c['order'] not in seen:
            seen.add(c['order'])
            unique.append(c)
    return aut_orbits, unique


def check_mult1(B, Gamma):
    """Return (all_mult1, decomp). Uses standard Sage character table pipeline."""
    ct = Gamma.character_table()
    classes = Gamma.conjugacy_classes()
    subrep = build_subrep(B, Gamma)
    chi = subrep_character(subrep, Gamma)
    decomp = irrep_decomposition(chi, Gamma, ct, classes)
    fs = frobenius_schur_indicators(Gamma, ct, classes, decomp)
    all_mult1 = all(mult == (2 if fs[idx] == -1 else 1) for idx, dim, mult in decomp)
    return all_mult1, decomp

## Libgap-safe overrides

For n ≥ 200, serializing permutations through pexpect overflows the GAP line buffer. We override `subrep_character` and `isotypic_projectors` to use libgap directly, reading image tuples via `ListPerm` instead of string serialization.

In [3]:
def _img(g_lgap, n):
    """0-indexed image tuple of a libgap permutation on n points."""
    return tuple(int(x) - 1 for x in libgap.ListPerm(g_lgap, n))


def subrep_character_libgap(subrep, aut_group):
    n = aut_group.degree()
    img_to_mat = {tuple(int(g(i)) for i in range(n)): M for g, M in subrep.items()}
    lgap_classes = libgap(aut_group).ConjugacyClasses()
    return [img_to_mat[_img(C.Representative(), n)].trace() for C in lgap_classes]


def isotypic_projectors_libgap(subrep, aut_group, ct, classes, paired_irreps):
    n = aut_group.degree()
    order = aut_group.order()
    d = next(iter(subrep.values())).nrows()
    R = next(iter(subrep.values())).base_ring()
    img_to_mat = {tuple(int(g(i)) for i in range(n)): M for g, M in subrep.items()}
    lgap_classes = libgap(aut_group).ConjugacyClasses()

    class_sums = []
    for C_lgap in lgap_classes:
        cs = matrix(R, d, d)
        for g_lgap in C_lgap.AsList():
            cs += img_to_mat[_img(g_lgap, n)]
        class_sums.append(cs)

    projectors = {}
    for group in paired_irreps:
        P = matrix(QQbar, d, d)
        for idx in group:
            chi_i = ct[idx]
            d_i = QQbar(chi_i[0])
            P_i = matrix(QQbar, d, d)
            for j, cs in enumerate(class_sums):
                P_i += QQbar(chi_i[j]).conjugate() * cs.change_ring(QQbar)
            P += (d_i / order) * P_i
        projectors[group[0]] = P
    return projectors

## Certification with a subgroup

Runs the full exact pipeline using Γ in place of Aut(G). The LP uses Γ-edge-orbits; the final certificate is verified against both Γ-orbits and Aut-orbits.

In [4]:
def certify_with_subgroup(G, which, Gamma, aut_orbits, B_override=None, use_libgap=False, verify=True):
    """
    Exact certification using Gamma <= Aut(G).
    B_override: supply a precomputed eigenspace basis (e.g. over QQ).
    use_libgap: use libgap-safe overrides for subrep_character and isotypic_projectors.
    verify: run verify_certificate as a sanity check (slow for large n; not needed for correctness).
    Returns (phi_vectors, weights, lam) on success, or None on failure.
    """
    gamma_orbits = get_edge_orbits(G, list(Gamma.gens()))

    t0 = time.time()
    if B_override is not None:
        lam = eigenspace_exact(G, which=which)[0]
        B = B_override
    else:
        lam, B = eigenspace_exact(G, which=which)
    print(f"  eigenspace: dim={B.nrows()}  ({time.time()-t0:.1f}s)")

    t0 = time.time()
    subrep = build_subrep(B, Gamma)
    print(f"  subrep: |Gamma|={Gamma.order()}  ({time.time()-t0:.1f}s)")

    ct = Gamma.character_table()
    classes = Gamma.conjugacy_classes()

    t0 = time.time()
    if use_libgap:
        chi = subrep_character_libgap(subrep, Gamma)
    else:
        chi = subrep_character(subrep, Gamma)
    decomp = irrep_decomposition(chi, Gamma, ct, classes)
    fs = frobenius_schur_indicators(Gamma, ct, classes, decomp)
    print(f"  decomp: {[(i,d,m) for i,d,m in decomp]}  ({time.time()-t0:.1f}s)")

    paired = pair_complex_conjugates(decomp, ct, fs)
    t0 = time.time()
    if use_libgap:
        projectors = isotypic_projectors_libgap(subrep, Gamma, ct, classes, paired)
    else:
        projectors = isotypic_projectors(subrep, Gamma, ct, classes, paired)
    phi_vectors = get_isotypic_representative(projectors, B)
    energies = orbit_energies(phi_vectors, gamma_orbits)
    print(f"  projectors + representatives  ({time.time()-t0:.1f}s)")

    t0 = time.time()
    weights = exact_weights(energies, gamma_orbits)
    if weights is None:
        print("  LP infeasible")
        return None
    print(f"  weights: {dict(weights)}  ({time.time()-t0:.1f}s)")

    # A feasible LP solution is an exact certificate by construction — verify is optional.
    if verify:
        phi = combine_certificate(phi_vectors, weights)
        ok_gamma = verify_certificate(phi, G, lam, gamma_orbits)
        ok_aut   = verify_certificate(phi, G, lam, aut_orbits)
        print(f"  verify Gamma-orbits: {ok_gamma}  |  Aut-orbits: {ok_aut}")
        if not ok_aut:
            return None

    return phi_vectors, weights, lam

## Special case: HoG 56133 (rational eigenspaces)

HoG 56133 has n=220 (pexpect overflow) and eigenspace dimensions 110 (λ₂=16) and 88 (λₙ=25). Since both eigenvalues are rational, we compute the eigenspace over ℚ rather than AA — this makes `build_subrep` orders of magnitude faster.

In [5]:
def eigenspace_qq(G, lam_rational):
    """Eigenspace basis over QQ for a known rational eigenvalue."""
    n = G.order()
    L = G.laplacian_matrix().change_ring(QQ)
    B = (L - QQ(lam_rational) * identity_matrix(QQ, n)).right_kernel().basis_matrix()
    return AA(QQ(lam_rational)), B

## Main loop

For each timeout graph: search for the smallest mult-1 subgroup, then certify. For n ≥ 200 we use libgap overrides throughout.

In [6]:
# Rational eigenvalues for graphs where we bypass AA eigenspace computation
RATIONAL_EIGENVALUES = {
    56133: {'lambda2': 16, 'lambdan': 25},
}

# verify_certificate is slow for large n since it evaluates exact rational arithmetic
# on the full phi vector; skip it for 42921 (n=234) where the LP feasibility is sufficient.
SKIP_VERIFY = {42921, 56133}

results = {}  # (hog_id, direction) -> (phi_vectors, weights, lam) or reason string

for row in timeout_rows:
    hog_id = int(row['id'])
    n = int(row['order'])
    use_libgap = (n >= 200)
    do_verify = (hog_id not in SKIP_VERIFY)

    directions = []
    if row['lcr_cert'] == 'timed_out':
        directions.append(('lcr', 'lambda2'))
    if row['ucr_cert'] == 'timed_out':
        directions.append(('ucr', 'lambdan'))

    print(f"\n{'='*60}")
    print(f"HoG {hog_id}  n={n}  directions={[d for d,_ in directions]}  libgap={use_libgap}  verify={do_verify}")

    G = Graph(g6_map[hog_id])
    G.relabel()

    t0 = time.time()
    aut_full = G.automorphism_group()
    print(f"  |Aut| = {aut_full.order()}  ({time.time()-t0:.1f}s)")

    t0 = time.time()
    aut_orbits, candidates = find_subgroup_candidates(G, aut_full, n_random=100, max_k=4)
    print(f"  {len(candidates)} subgroup candidates  ({time.time()-t0:.1f}s)")
    for c in candidates[:8]:
        print(f"    {c['name']:<22}  |Γ|={c['order']:>10}  #orbits={len(c['orbits'])}")

    for direction, which in directions:
        print(f"\n  -- {direction} ({which}) --")

        # Precompute eigenspace (QQ override for known rational eigenvalues)
        rat_eigs = RATIONAL_EIGENVALUES.get(hog_id, {})
        if which in rat_eigs:
            print(f"  using QQ eigenspace (lam={rat_eigs[which]})")
            lam_aa, B_qq = eigenspace_qq(G, rat_eigs[which])
            B_for_mult1 = B_qq
        else:
            lam_aa, B_for_mult1 = eigenspace_exact(G, which=which)
            B_qq = None
        print(f"  eigenspace dim={B_for_mult1.nrows()}")

        # Find smallest mult-1 subgroup
        chosen = None
        for c in candidates:
            try:
                all_m1, decomp = check_mult1(B_for_mult1, c['group'])
                status = 'mult-1 ✓' if all_m1 else 'mult issues'
                print(f"    {c['name']:<22}  |Γ|={c['order']:>8}  {status}")
                if all_m1 and chosen is None:
                    chosen = c
                    break
            except Exception as e:
                print(f"    {c['name']:<22}  error: {e}")

        if chosen is None:
            print("  no mult-1 candidate found")
            results[(hog_id, direction)] = 'no_mult1_candidate'
            continue

        print(f"  certifying with {chosen['name']}  |Γ|={chosen['order']} ...")
        t0 = time.time()
        try:
            result = certify_with_subgroup(
                G, which, chosen['group'], aut_orbits,
                B_override=B_qq,
                use_libgap=use_libgap,
                verify=do_verify,
            )
            elapsed = time.time() - t0
            if result is not None:
                print(f"  CERTIFIED in {elapsed:.1f}s")
                results[(hog_id, direction)] = result
            else:
                print(f"  FAILED in {elapsed:.1f}s")
                results[(hog_id, direction)] = 'failed'
        except Exception as e:
            print(f"  ERROR: {e}")
            results[(hog_id, direction)] = f'error: {e}'

print(f"\n{'='*60}")
print("Summary:")
for (hid, direction), val in results.items():
    status = 'exact' if isinstance(val, tuple) else val
    print(f"  HoG {hid:5d} ({direction}): {status}")


HoG 32782  n=12  directions=['ucr']  libgap=False  verify=True
  |Aut| = 518400  (0.2s)
  25 subgroup candidates  (5.7s)
    sylow/p=5               |Γ|=        25  #orbits=7
    sylow/p=3               |Γ|=        81  #orbits=7
    rand/k=2,s=1            |Γ|=       120  #orbits=4
    sylow/p=2               |Γ|=       256  #orbits=8
    rand/k=2,s=35           |Γ|=       480  #orbits=3
    rand/k=2,s=33           |Γ|=       720  #orbits=3
    rand/k=2,s=60           |Γ|=      1080  #orbits=4
    rand/k=2,s=31           |Γ|=      1200  #orbits=6

  -- ucr (lambdan) --
  eigenspace dim=6
    sylow/p=5               |Γ|=      25  mult issues
    sylow/p=3               |Γ|=      81  mult issues
    rand/k=2,s=1            |Γ|=     120  mult issues
    sylow/p=2               |Γ|=     256  mult issues
    rand/k=2,s=35           |Γ|=     480  mult-1 ✓
  certifying with rand/k=2,s=35  |Γ|=480 ...
  eigenspace: dim=6  (0.0s)
  subrep: |Gamma|=480  (0.4s)
  decomp: [(0, 1, 1), (20, 5, 1)] 

## Update CSV and JSONL

Write certified results back to `exact_certs.csv` and append serialized certificates to `exact_certs.jsonl`.

In [7]:
with open(PHI_JSONL, 'a') as f_phi:
    for (hog_id, direction), val in results.items():
        row = all_rows[hog_id]
        cert_col = f'{direction}_cert'

        if isinstance(val, tuple):
            phi_vectors, weights, lam = val
            row[cert_col] = 'exact'
            serialized = serialize_certificate(phi_vectors, weights, lam)
            f_phi.write(json.dumps({
                'id':  hog_id,
                'lcr': serialized if direction == 'lcr' else None,
                'ucr': serialized if direction == 'ucr' else None,
            }) + '\n')
        elif val == 'failed' or val == 'no_mult1_candidate':
            row[cert_col] = 'mult_issue'
        elif val.startswith('error'):
            row[cert_col] = 'error'
        # leave as 'timed_out' if not in results

with open(OUTPUT_CSV, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(all_rows.values())

print(f"Updated {OUTPUT_CSV} and appended to {PHI_JSONL}.")

Updated exact_certs.csv and appended to exact_certs.jsonl.
